# Will this customer leave?

A service keeps a row for each customer. Some rows are marked leave, and the others are marked stay. The question is which mark belongs on a customer whose mark you are not allowed to see.

This path uses the IBM telco customer churn sample: 7043 customers and 21 columns, with the label `Churn` equal to `Yes` or `No`. The file is `data/Telco-Customer-Churn.csv`. The source note is `data/SOURCE.txt`, and the Apache License 2.0 text is `data/LICENSE`. These lessons are not an IBM product, and they are not endorsed by IBM.

Accuracy counts every correct mark, stay or leave. Recall of leaving counts only the leavers you flagged. With 10 customers and 3 leavers, the constant prediction Stay is correct on the 7 who stay:

$$
\mathrm{accuracy} = \dfrac{7}{10}.
$$

That prediction flags 0 leavers, so its recall is 0.

A second rule flags 4 customers. Two of them leave, and two of them stay. It misses 1 leaver, and it correctly keeps 5 stayers. Then

$$
\mathrm{precision} = \dfrac{2}{2+2} = \dfrac{1}{2},
\qquad
\mathrm{recall} = \dfrac{2}{2+1} = \dfrac{2}{3},
$$

$$
F_1 = \dfrac{2 \cdot \dfrac{1}{2} \cdot \dfrac{2}{3}}{\dfrac{1}{2}+\dfrac{2}{3}} = \dfrac{\dfrac{2}{3}}{\dfrac{7}{6}} = \dfrac{4}{7}.
$$

The accuracy of that rule is also $7/10$. The two rules tie on accuracy. $F_1$ separates them: $4/7$ against a recall of 0.


In [1]:
# Hand arithmetic for the 10-customer sketch, then the size of the real table.
from fractions import Fraction

precision = Fraction(2, 4)
recall = Fraction(2, 3)
f1 = 2 * precision * recall / (precision + recall)
accuracy = Fraction(2 + 5, 10)
majority = Fraction(7, 10)
print(precision, recall, f1, accuracy, majority)
assert precision == Fraction(1, 2)
assert recall == Fraction(2, 3)
assert f1 == Fraction(4, 7)
assert accuracy == majority == Fraction(7, 10)


1/2 2/3 4/7 7/10 7/10


In [2]:
# Locate the telco table beside this notebook, or under the course root.
import csv
from pathlib import Path

def data_path():
    here = Path.cwd()
    name = "Telco-Customer-Churn.csv"
    for folder in [here, *here.parents]:
        nested = folder / "16-Customer Churn" / "data" / name
        direct = folder / "data" / name
        if nested.exists():
            return nested
        if direct.exists():
            return direct
    raise FileNotFoundError(name)

def read_customers():
    with data_path().open(newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))
# The sample really does contain 7043 customers and 1869 leavers.
rows = read_customers()
left = sum(row["Churn"] == "Yes" for row in rows)
stayed = sum(row["Churn"] == "No" for row in rows)
print(len(rows), left, stayed)
assert len(rows) == 7043
assert left == 1869 and stayed == 5174
assert left + stayed == 7043


7043 1869 5174


## Pitfall

A score that only reports accuracy can crown Stay-for-everyone. On the sketch, that rule and the useful rule both score $7/10$. The useful rule is the one with recall $2/3$ and $F_1 = 4/7$.

## Summary

Leaving is the mark `Churn = Yes`. Accuracy of Stay-for-everyone equals the stay rate. $F_1$ uses precision and recall of the leave mark. The file holds 7043 customers, 1869 of whom leave.
